# 02 — Backtest EMACross

Run our EMACross strategy against BTC-USD-PERP 1h data on a simulated Hyperliquid venue.

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + EMA overlay with trade markers, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweep — fast/slow grid with heatmap

**Prerequisites:** Run `scripts/fetch_hl_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import LoggingConfig
from nautilus_trader.indicators import ExponentialMovingAverage
from nautilus_trader.model.currencies import USDC
from nautilus_trader.model.data import BarType
from nautilus_trader.model.enums import AccountType, OmsType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.core import bar_type_str
from src.strategies.ema_cross import EMACross, EMACrossConfig

from charts import plot_ema_cross

# ── Shared config ────────────────────────────────────────────────
CATALOG_PATH    = "../data/catalog"
INSTRUMENT_ID   = "BTC-USD-PERP.HYPERLIQUID"
BAR_TYPE_STR    = bar_type_str(INSTRUMENT_ID, "1h")
VENUE           = Venue("HYPERLIQUID")
STARTING_CAPITAL = 100_000
TRADE_SIZE      = Decimal("0.01")   # 0.01 BTC per trade (~$950 notional at $95k)
FAST_EMA        = 20
SLOW_EMA        = 50
FAST_PERIODS = [5, 10, 15, 20]
SLOW_PERIODS = [30, 50, 75, 100]

print("Imports OK")

In [ ]:
# ── Cell 2: Load data from catalog ────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

print(f"Instrument : {instrument.id}")
print(f"Bar count  : {len(bars):,}")
print(f"First bar  : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar   : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────
engine = BacktestEngine(config=BacktestEngineConfig(
    logging=LoggingConfig(log_level="ERROR"),
))

engine.add_venue(
    venue=VENUE,
    oms_type=OmsType.NETTING,
    account_type=AccountType.MARGIN,
    base_currency=None,              # Multi-currency — settlement from instrument
    starting_balances=[Money(STARTING_CAPITAL, USDC)],
)

engine.add_instrument(instrument)
engine.add_data(bars)
print("Engine configured.")

In [ ]:
# ── Cell 4: Add EMACross strategy + run ───────────────────────────
config = EMACrossConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_size=TRADE_SIZE,
    fast_ema_period=FAST_EMA,
    slow_ema_period=SLOW_EMA,
)
strategy = EMACross(config=config)
engine.add_strategy(strategy)
engine.run()
print("Backtest complete.")

In [ ]:
# ── Cell 5: Reports ───────────────────────────────────────────────
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")
print()

print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 6: Price chart with EMA overlay + trade markers ──────────
#
# Recompute EMA values from bars for plotting (the strategy's internal
# indicator state isn't directly accessible after the run).

fast_ema_plot = ExponentialMovingAverage(FAST_EMA)
slow_ema_plot = ExponentialMovingAverage(SLOW_EMA)

timestamps, closes, fast_vals, slow_vals = [], [], [], []
for bar in bars:
    fast_ema_plot.handle_bar(bar)
    slow_ema_plot.handle_bar(bar)
    timestamps.append(pd.Timestamp(bar.ts_event, unit="ns", tz="UTC"))
    closes.append(float(bar.close))
    fast_vals.append(fast_ema_plot.value if fast_ema_plot.initialized else np.nan)
    slow_vals.append(slow_ema_plot.value if slow_ema_plot.initialized else np.nan)

price_df = pd.DataFrame({
    "close": closes,
    f"EMA({FAST_EMA})": fast_vals,
    f"EMA({SLOW_EMA})": slow_vals,
}, index=timestamps)

# ── Plot ──
fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(price_df.index, price_df["close"], label="Close", color="#555555", alpha=0.6, linewidth=0.8)
ax.plot(price_df.index, price_df[f"EMA({FAST_EMA})"], label=f"EMA({FAST_EMA})", linewidth=1.3)
ax.plot(price_df.index, price_df[f"EMA({SLOW_EMA})"], label=f"EMA({SLOW_EMA})", linewidth=1.3)

# Overlay trade entry markers from fills report
if not fills_report.empty:
    fr = fills_report.copy()
    price_col = "last_px" if "last_px" in fr.columns else "avg_px" if "avg_px" in fr.columns else None
    side_col = "side" if "side" in fr.columns else "order_side" if "order_side" in fr.columns else None

    if price_col and side_col and "ts_last" in fr.columns:
        fr["_px"] = fr[price_col].astype(float)
        fr["_ts"] = pd.to_datetime(fr["ts_last"].astype("int64"), unit="ns", utc=True)
        buys  = fr[fr[side_col].astype(str).str.contains("BUY", case=False)]
        sells = fr[fr[side_col].astype(str).str.contains("SELL", case=False)]
        ax.scatter(buys["_ts"], buys["_px"], marker="^", s=50, c="#2ecc71", label="Buy", zorder=5)
        ax.scatter(sells["_ts"], sells["_px"], marker="v", s=50, c="#e74c3c", label="Sell", zorder=5)

ax.set_title(f"BTC-USD-PERP  —  EMACross({FAST_EMA}/{SLOW_EMA})", fontsize=13)
ax.set_xlabel("Time")
ax.set_ylabel("Price (USD)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 6a: Price chart ───────────────────────────────────────────

fig = plot_ema_cross(
    bars, fills_report,
    fast_period=FAST_EMA,
    slow_period=SLOW_EMA,
    instrument_label=INSTRUMENT_ID,
    bar_label="1h",
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 7: Equity curve ──────────────────────────────────────────
#
# The analyzer's returns() gives per-period returns. If it returns
# None or empty, fall back to the account report for a balance curve.

analyzer = engine.portfolio.analyzer

fig, ax = plt.subplots(figsize=(14, 5))
plotted = False

# Method 1: Cumulative returns from analyzer
try:
    returns = analyzer.returns()
    if returns is not None and len(returns) > 0:
        cumulative = (1 + returns).cumprod()
        cumulative.plot(ax=ax, label="Cumulative Return")
        plotted = True
except Exception:
    pass

# Method 2: Fallback to account balance curve
if not plotted and account_report is not None and not account_report.empty:
    account_report.plot(ax=ax, label="Account Balance")
    ax.set_ylabel("Balance (USDC)")
    plotted = True

if plotted:
    ax.set_title(f"Equity Curve — EMACross({FAST_EMA}/{SLOW_EMA})  BTC 1h", fontsize=13)
    ax.set_xlabel("Time")
    ax.grid(True, alpha=0.2)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No returns or account data available for equity curve.")

In [ ]:
# ── Cell 8: Summary statistics ────────────────────────────────────
#
# The Rust-ported analyzer in 1.223.0 requires calculate_statistics()
# before the get_performance_stats_*() methods return data.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.positions()

analyzer.calculate_statistics(account, positions)

general_stats = analyzer.get_performance_stats_general()
pnl_stats     = analyzer.get_performance_stats_pnls(USDC)
returns_stats  = analyzer.get_performance_stats_returns()

print("=== General ===")
for k, v in general_stats.items():
    print(f"  {k}: {v}")

print("\n=== PnL (USDC) ===")
for k, v in pnl_stats.items():
    print(f"  {k}: {v}")

print("\n=== Returns ===")
for k, v in returns_stats.items():
    print(f"  {k}: {v}")

print(f"\nTotal PnL      : {analyzer.total_pnl(USDC)}")
print(f"Total PnL %    : {analyzer.total_pnl_percentage(USDC)}")

In [ ]:
# ── Cell 9: Parameter sweep ───────────────────────────────────────
#
# Grid search over fast/slow EMA period combinations.
# engine.reset() preserves instruments and data — only strategies and
# execution state are cleared. No need for clear_strategies().

def run_single_backtest(eng, fast: int, slow: int) -> dict:
    """Reset engine, run one parameter combo, extract stats."""
    eng.reset()

    cfg = EMACrossConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_size=TRADE_SIZE,
        fast_ema_period=fast,
        slow_ema_period=slow,
    )
    eng.add_strategy(EMACross(cfg))
    eng.run()

    a = eng.portfolio.analyzer
    acct = eng.cache.account_for_venue(VENUE)
    pos = eng.cache.positions()
    a.calculate_statistics(acct, pos)

    row = {
        "fast": fast,
        "slow": slow,
        "total_pnl": float(a.total_pnl(USDC)),
        "total_pnl_pct": float(a.total_pnl_percentage(USDC)),
    }

    # Pull detailed stats — keys vary by NT version so we merge dicts
    try:
        for k, v in a.get_performance_stats_general().items():
            row[k] = v
    except Exception as e:
        print(f"  Warning: general stats failed for {fast}/{slow}: {e}")

    try:
        for k, v in a.get_performance_stats_pnls(USDC).items():
            row[k] = v
    except Exception as e:
        print(f"  Warning: PnL stats failed for {fast}/{slow}: {e}")

    try:
        for k, v in a.get_performance_stats_returns().items():
            row[k] = v
    except Exception as e:
        print(f"  Warning: returns stats failed for {fast}/{slow}: {e}")

    return row


# ── Run the sweep ──
results = []
total = len(FAST_PERIODS) * len(SLOW_PERIODS)
count = 0

for fast in FAST_PERIODS:
    for slow in SLOW_PERIODS:
        count += 1
        row = run_single_backtest(engine, fast, slow)
        results.append(row)
        print(
            f"  [{count}/{total}] fast={fast:>3}, slow={slow:>3}  "
            f"PnL={row['total_pnl']:>10.2f}  PnL%={row['total_pnl_pct']:>7.2f}%"
        )

results_df = pd.DataFrame(results)

# Show key columns (some stat keys may vary — show what's available)
display_cols = ["fast", "slow", "total_pnl", "total_pnl_pct"]
# Add any Sharpe / drawdown / win-rate columns the analyzer produced
for col in results_df.columns:
    lower = col.lower()
    if any(kw in lower for kw in ["sharpe", "drawdown", "win", "trades", "expectancy", "factor"]):
        display_cols.append(col)

display_cols = [c for c in display_cols if c in results_df.columns]
print("\n=== Parameter Sweep Results ===")
display(results_df[display_cols])

In [ ]:
# ── Cell 10: PnL heatmap ──────────────────────────────────────────
pivot = results_df.pivot(index="slow", columns="fast", values="total_pnl")

fig, ax = plt.subplots(figsize=(8, 6))

# Diverging colourmap centred at 0
vmax = max(abs(pivot.values.min()), abs(pivot.values.max()))
im = ax.imshow(
    pivot.values,
    cmap="RdYlGn",
    aspect="auto",
    vmin=-vmax,
    vmax=vmax,
)

# Labels
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([str(i) for i in pivot.index])
ax.set_xlabel("Fast EMA Period")
ax.set_ylabel("Slow EMA Period")
ax.set_title("Total PnL (USDC) by EMA Parameters")

# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        color = "white" if abs(val) > vmax * 0.6 else "black"
        ax.text(j, i, f"{val:,.0f}", ha="center", va="center", fontsize=10, color=color)

fig.colorbar(im, ax=ax, label="PnL (USDC)")
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 11: Cleanup ──────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")